# Price Estimator with Frontier Model

In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv

In [ ]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
print(test[0].summary)

In [ ]:
messages_for(test[0])

In [ ]:
# The function for gpt-4.1-nano

def luna(item):
    response = completion(model="openai/gpt-5.6-luna", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
luna(test[0])

In [ ]:
test[0].price

In [ ]:
evaluate(luna, test)